# EEG_07e — Pre-costruzione Tensori Grafo / Ipergrafo da CSV

Legge i CSV di Paolo (`data/raw_csv/training_set/PXXX_SYYY/parola_img.csv`)
e costruisce tensori PyG `Data(x, edge_index, y, subj, sess)` in `data/interim/graphs/`.

**Struttura CSV**: 61 righe × 384 colonne (canali × campioni), nessun header.

Supporta:
- Metodi connettività: **PCC**, **PLV**, **wPLI**
- K valori per k-NN
- **Pruning**: soglia edge + rimozione canali a bassa connettività
- Costruzione **ipergrafo** (per EEG_11)

In [ ]:
# ============================================================
# CONFIGURAZIONE
# ============================================================

# Metodi connettività — grafi
METHODS = ["pcc"]           # "pcc" | "plv" | "wpli"

# K-vicini per k-NN graph
K_VALUES = [6]

# Soglia edge: rimuove archi con peso < threshold (0.0 = nessuna)
EDGE_THRESHOLD = 0.0

# Pruning canali: rimuove canali con connettività media < mean - sigma*std
# None = nessun pruning
CHANNEL_PRUNING_SIGMA = None

# Canali da escludere esplicitamente (es. ["A1", "A2"] o [])
CHANNEL_DROP_NAMES = []

# Costruisci anche ipergrafo (per EEG_11)
BUILD_HGNN = True
METHODS_HGNN = ["pcc"]     # metodi connettività per ipergrafi
K_HYPER      = 6

# Schema label da applicare
CLUSTER_SCHEME = "concr4"   # "concr4" | "ward4" | "sem5" | "pos4" | "raw110"

# Forza ricostruzione anche se file già esiste
FORCE_REBUILD = True

print("Config OK")

In [ ]:
# ============================================================
# IMPORT E PATHS
# ============================================================

import os, sys, json
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import pandas as pd
import torch

torch.multiprocessing.set_sharing_strategy('file_system')  # evita OOM su /dev/shm condivisa
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from torch_geometric.data import Data

# Trova project root
project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents)
     if (p / ".git").exists()),
    Path().resolve()
)
sys.path.insert(0, str(project_root / "scripts"))
from utils import load_label_scheme

# Paths
CSV_ROOT   = project_root / "data" / "raw_csv" / "training_set"
GRAPHS_DIR = project_root / "data" / "interim" / "graphs"
CONFIGS    = project_root / "configs" / "label_schemes"
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

N_CHANS   = 61
N_SAMPLES = 384

# Mapping parola → label_id (da label2idx.json)
with open(CONFIGS / "label2idx.json") as f:
    word2labelid = json.load(f)   # {"accendere": 0, "acqua": 1, ...}

# Schema cluster
labelid2cluster, N_CLASSES, cluster_names = load_label_scheme(CLUSTER_SCHEME, project_root / "data" / "interim")

# Lista cartelle soggetto-sessione
session_dirs = sorted(CSV_ROOT.iterdir())
print(f"Project root  : {project_root}")
print(f"CSV root      : {CSV_ROOT}")
print(f"Cartelle sess : {len(session_dirs)}")  # ~357
print(f"N_CHANS       : {N_CHANS}")
print(f"Schema        : {CLUSTER_SCHEME} ({N_CLASSES} classi)")
print(f"Esempio dir   : {session_dirs[0].name}")

In [ ]:
# ============================================================
# PARSING NOMI CARTELLA
# PXXX_SYYY → subj_id (int), sess_id (int)
# ============================================================

def parse_folder(folder_name: str):
    """Es. 'P003_S002' → (3, 2)"""
    parts = folder_name.split("_")
    subj = int(parts[0][1:])   # rimuove 'P'
    sess = int(parts[1][1:])   # rimuove 'S'
    return subj, sess

def load_csv_trial(csv_path: Path) -> np.ndarray:
    """Legge CSV 61×384 senza header. Restituisce array float32 (61, 384)."""
    return pd.read_csv(csv_path, header=None).values.astype(np.float32)

# Test
test_dir = session_dirs[0]
test_csv = sorted(test_dir.iterdir())[0]
x_test   = load_csv_trial(test_csv)
subj_t, sess_t = parse_folder(test_dir.name)
word_t   = test_csv.stem.replace("_img", "")

print(f"Cartella: {test_dir.name} → subj={subj_t}, sess={sess_t}")
print(f"File    : {test_csv.name} → parola='{word_t}', label_id={word2labelid.get(word_t, '??')}")
print(f"Shape   : {x_test.shape}  (atteso: (61, 384))")
assert x_test.shape == (N_CHANS, N_SAMPLES), f"Shape attesa ({N_CHANS}, {N_SAMPLES}), trovata {x_test.shape}"
print("✅ Parsing OK")

In [ ]:
# ============================================================
# FUNZIONI DI CONNETTIVITÀ
# ============================================================

def pcc_matrix(x_np: np.ndarray) -> np.ndarray:
    """Pearson |PCC| tra canali. Shape: (N, N)"""
    pcc = np.abs(np.corrcoef(x_np))
    np.fill_diagonal(pcc, 0.0)
    return pcc


def plv_matrix(x_np: np.ndarray) -> np.ndarray:
    """Phase Locking Value via Hilbert. Shape: (N, N)"""
    from scipy.signal import hilbert
    N = x_np.shape[0]
    phases = np.angle(hilbert(x_np, axis=1))
    plv = np.zeros((N, N))
    for i in range(N):
        for j in range(i + 1, N):
            diff = phases[i] - phases[j]
            plv[i, j] = plv[j, i] = np.abs(np.mean(np.exp(1j * diff)))
    return plv


def wpli_matrix(x_np: np.ndarray) -> np.ndarray:
    """Weighted Phase Lag Index. Shape: (N, N)"""
    from scipy.signal import hilbert
    N = x_np.shape[0]
    analytic = hilbert(x_np, axis=1)
    wpli = np.zeros((N, N))
    for i in range(N):
        for j in range(i + 1, N):
            cs = analytic[i] * np.conj(analytic[j])
            im = np.imag(cs)
            w  = np.abs(im)
            wpli[i, j] = wpli[j, i] = np.abs(np.mean(im * w)) / (np.mean(w) + 1e-9)
    return wpli


CONN_FN = {"pcc": pcc_matrix, "plv": plv_matrix, "wpli": wpli_matrix}


def knn_edge_index(matrix: np.ndarray, k: int,
                   threshold: float = 0.0) -> torch.LongTensor:
    """k-NN graph da matrice connettività. Restituisce edge_index (2, E)."""
    N = matrix.shape[0]
    rows, cols = [], []
    for i in range(N):
        row = matrix[i].copy(); row[i] = -1.0
        if threshold > 0.0:
            row[row < threshold] = 0.0
        top_k = np.argsort(row)[-k:]
        for j in top_k:
            if row[j] > 0.0 or threshold == 0.0:
                rows += [i, j]; cols += [j, i]
    return torch.tensor([rows, cols], dtype=torch.long)


def hyperedge_index_fn(x_np: np.ndarray, k: int,
                       method: str = "pcc",
                       threshold: float = 0.0) -> torch.LongTensor:
    """
    Ipergrafo k-NN per-trial.
    Ogni nodo i è centro di un'iperedge con i + top-k vicini.

    Args:
        x_np   : segnale (N, T)
        k      : vicini per iperedge
        method : "pcc" | "plv" | "wpli" — metrica di connettività
        threshold: rimuovi vicini con peso < threshold
    """
    mat = CONN_FN[method](x_np)
    N   = mat.shape[0]
    vertex_list, edge_list = [], []
    for e_id in range(N):
        row = mat[e_id].copy()
        if threshold > 0.0:
            row[row < threshold] = 0.0
        top_k   = np.argsort(row)[-k:]
        members = [e_id] + [j for j in top_k
                            if (row[j] > 0.0 or threshold == 0.0)]
        for v in members:
            vertex_list.append(v); edge_list.append(e_id)
    return torch.tensor([vertex_list, edge_list], dtype=torch.long)


print("Funzioni connettività OK")

In [ ]:
# ============================================================
# FUNZIONE HELPER: itera tutti i trial da CSV
# Usa sempre tutti i 61 canali — nessun pruning qui.
# Il pruning viene applicato dopo la build in cell 11-12.
# ============================================================

def iter_all_trials():
    """
    Generator: itera tutte le cartelle PXXX_SYYY e tutti i CSV.
    Yields: (x_np, label_id, cluster_id, subj_id, sess_id)
    Salta file con parola non in label2idx o label_id non in labelid2cluster.
    Restituisce sempre tutti i 61 canali.
    """
    for sess_dir in sorted(CSV_ROOT.iterdir()):
        if not sess_dir.is_dir():
            continue
        subj_id, sess_id = parse_folder(sess_dir.name)
        for csv_path in sorted(sess_dir.iterdir()):
            if csv_path.suffix != ".csv":
                continue
            word = csv_path.stem.replace("_img", "")
            if word not in word2labelid:
                continue
            label_id = word2labelid[word]
            if label_id not in labelid2cluster:
                continue
            cluster_id = labelid2cluster[label_id]
            x_np = load_csv_trial(csv_path)   # (61, 384) — tutti i canali
            yield x_np, label_id, cluster_id, subj_id, sess_id

# Conta trial totali
n_total = sum(1 for _ in iter_all_trials())
print(f"Trial totali validi: {n_total}")
print(f"(atteso ~{len(session_dirs) * 110} = {len(session_dirs)} sess × 110 parole)")

In [ ]:
# ============================================================
# BUILD GRAPH TENSORS (PCC / PLV / wPLI)
# Output: data/interim/graphs/graph_{method}_k{k}.pt
# ============================================================

CONN_FN = {"pcc": pcc_matrix, "plv": plv_matrix, "wpli": wpli_matrix}

for method in METHODS:
    for k in K_VALUES:
        out_path = GRAPHS_DIR / f"graph_{method}_k{k}.pt"
        if out_path.exists() and not FORCE_REBUILD:
            print(f"Skip: {out_path.name}")
            continue

        print(f"\nCostruendo {out_path.name}...")
        conn_fn   = CONN_FN[method]
        data_list = []

        for x_np, label_id, cluster_id, subj_id, sess_id in tqdm(
                iter_all_trials(), total=n_total, desc=f"{method} k={k}"):

            matrix     = conn_fn(x_np)
            edge_index = knn_edge_index(matrix, k=k, threshold=EDGE_THRESHOLD)

            data_list.append(Data(
                x          = torch.tensor(x_np, dtype=torch.float32),
                edge_index = edge_index,
                y          = torch.tensor(cluster_id, dtype=torch.long),
                label_id   = torch.tensor(label_id,   dtype=torch.long),
                subj       = torch.tensor(subj_id,    dtype=torch.long),
                sess       = torch.tensor(sess_id,    dtype=torch.long),
            ))

        torch.save(data_list, out_path)
        print(f"  ✅ Salvato: {out_path.name} — {len(data_list)} grafi  x.shape={data_list[0].x.shape}")

In [ ]:
# ============================================================
# BUILD HYPERGRAPH TENSORS (PCC / PLV / wPLI)
# Output: data/interim/graphs/hgraph_{method}_k{k}.pt
# ============================================================

if BUILD_HGNN:
    for method in METHODS_HGNN:
        for k in [K_HYPER]:
            out_path = GRAPHS_DIR / f"hgraph_{method}_k{k}.pt"
            if out_path.exists() and not FORCE_REBUILD:
                print(f"Skip: {out_path.name}")
                continue

            print(f"\nCostruendo {out_path.name}...")
            data_list = []

            for x_np, label_id, cluster_id, subj_id, sess_id in tqdm(
                    iter_all_trials(), total=n_total,
                    desc=f"hgraph {method} k={k}"):

                he_index     = hyperedge_index_fn(x_np, k=k,
                                                  method=method,
                                                  threshold=EDGE_THRESHOLD)
                n_hyperedges = he_index[1].max().item() + 1

                data_list.append(Data(
                    x               = torch.tensor(x_np, dtype=torch.float32),
                    hyperedge_index = he_index,
                    num_hyperedges  = n_hyperedges,
                    y               = torch.tensor(cluster_id, dtype=torch.long),
                    label_id        = torch.tensor(label_id,   dtype=torch.long),
                    subj            = torch.tensor(subj_id,    dtype=torch.long),
                    sess            = torch.tensor(sess_id,    dtype=torch.long),
                ))

            torch.save(data_list, out_path)
            print(f"  ✅ Salvato: {out_path.name} — {len(data_list)} ipergrafi")

In [ ]:
# ============================================================
# SANITY CHECK + COMPARISON PLOT GRAFO vs IPERGRAFO
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

print(f"=== File in {GRAPHS_DIR} ===")
graph_data  = {}
hgraph_data = {}

for pt_file in sorted(GRAPHS_DIR.glob("*.pt")):
    dl = torch.load(pt_file, weights_only=False)
    d0 = dl[0]
    subj_ids = sorted(set(d.subj.item() for d in dl))
    y_vals   = [d.y.item() for d in dl]

    is_hgraph = pt_file.name.startswith("hgraph")

    if not is_hgraph:
        ei = d0.edge_index
        print(f"  {pt_file.name}: {len(dl)} grafi | "
              f"x={d0.x.shape} | edge_index={ei.shape if ei is not None else 'None'} | "
              f"n_subj={len(subj_ids)} | y_range=[{min(y_vals)},{max(y_vals)}]")
        graph_data[pt_file.stem] = dl
    else:
        he = d0.hyperedge_index
        print(f"  {pt_file.name}: {len(dl)} ipergrafi | "
              f"x={d0.x.shape} | he={he.shape if he is not None else 'None'} | "
              f"n_he={d0.num_hyperedges} | y_range=[{min(y_vals)},{max(y_vals)}]")
        hgraph_data[pt_file.stem] = dl

if not graph_data and not hgraph_data:
    print("⚠️  Nessun file .pt trovato — esegui prima le celle build (6-7).")
else:
    print(f"\n✅ Sanity check OK — {N_CHANS} canali, schema={CLUSTER_SCHEME}")

# ============================================================
# COMPARISON PLOT: grafo vs ipergrafo su trial campione
# ============================================================

if graph_data and hgraph_data:
    import networkx as nx

    g_key  = list(graph_data.keys())[0]
    hg_key = list(hgraph_data.keys())[0]
    d_g    = graph_data[g_key][0]
    d_hg   = hgraph_data[hg_key][0]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # --- 1. PCC heatmap del trial campione ---
    x_np = d_g.x.numpy()
    pcc  = np.abs(np.corrcoef(x_np))
    np.fill_diagonal(pcc, 0)
    im = axes[0].imshow(pcc, cmap="RdYlBu_r", vmin=0, vmax=0.5)
    axes[0].set_title("PCC matrice\n(trial campione)")
    axes[0].set_xlabel("Canale"); axes[0].set_ylabel("Canale")
    plt.colorbar(im, ax=axes[0])

    # --- 2. Grafo k-NN ---
    G = nx.Graph()
    G.add_nodes_from(range(d_g.x.shape[0]))
    ei = d_g.edge_index.numpy()
    for s, t in zip(ei[0], ei[1]):
        if s < t:
            G.add_edge(int(s), int(t))
    deg_g = [G.degree(n) for n in G.nodes()]
    pos   = nx.circular_layout(G)
    nx.draw_networkx(G, pos=pos, ax=axes[1],
                     node_size=80, node_color=deg_g, cmap="viridis",
                     with_labels=False, width=0.4, edge_color="gray")
    axes[1].set_title(f"Grafo k-NN ({g_key})\n"
                      f"{G.number_of_nodes()} nodi, {G.number_of_edges()} archi\n"
                      f"grado medio={np.mean(deg_g):.1f}")

    # --- 3. Ipergrafo: grado iperedge per nodo ---
    he = d_hg.hyperedge_index.numpy()
    n_nodes    = d_hg.x.shape[0]
    n_he       = d_hg.num_hyperedges
    node_deg   = np.bincount(he[0], minlength=n_nodes)
    he_size    = np.bincount(he[1], minlength=n_he)

    axes[2].bar(range(n_nodes), node_deg, color="steelblue", alpha=0.7, label="Grado nodo")
    ax2t = axes[2].twinx()
    ax2t.hist(he_size, bins=20, color="orange", alpha=0.5, label="Dim. iperedge")
    axes[2].set_xlabel("Nodo (canale)")
    axes[2].set_ylabel("Grado nodo (n° iperedge)", color="steelblue")
    ax2t.set_ylabel("Freq. dim. iperedge", color="orange")
    axes[2].set_title(f"Ipergrafo ({hg_key})\n"
                      f"{n_nodes} nodi, {n_he} iperedge\n"
                      f"dim.media={he_size.mean():.1f}")
    axes[2].legend(loc="upper left"); ax2t.legend(loc="upper right")

    plt.tight_layout()
    fig_path = project_root / "figures" / "eeg07e_graph_vs_hypergraph.png"
    fig.savefig(fig_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Plot salvato: {fig_path}")

In [ ]:
# ============================================================
# ANALISI CONNETTIVITÀ — tutti i soggetti / sessioni / parole
# Usa METHODS definiti in cella 1 — guida al pruning
# ============================================================

import seaborn as sns

# wPLI normalizzato [0,1] per confronto diretto con PCC/PLV
def wpli_matrix_norm(x_np):
    m = wpli_matrix(x_np)
    mn, mx = m.min(), m.max()
    return (m - mn) / (mx - mn + 1e-9)

CONN_FN_ANALYSIS = {"pcc": pcc_matrix, "plv": plv_matrix, "wpli": wpli_matrix_norm}

# Ogni N trial campiona i valori per l'istogramma (risparmio RAM)
VAL_SUBSAMPLE = 10   # prendi vals 1 trial su VAL_SUBSAMPLE

# ---- Scansione piatta di tutti i trial validi ----
_subjects_all = sorted(set(d.name.split("_")[0]
                           for d in CSV_ROOT.iterdir() if d.is_dir()))
_sessions_all = sorted(set(d.name.split("_")[1]
                           for d in CSV_ROOT.iterdir() if d.is_dir()))
_words_all    = sorted(word2labelid.keys())

print("Scansione trial...")
_trial_paths = [
    CSV_ROOT / f"{s}_{ss}" / f"{w}_img.csv"
    for s in _subjects_all
    for ss in _sessions_all
    for w in _words_all
    if (CSV_ROOT / f"{s}_{ss}" / f"{w}_img.csv").exists()
]
n_trials_analysis = len(_trial_paths)
print(f"Trial trovati  : {n_trials_analysis}")
print(f"Metodi         : {METHODS}")
if "wpli" in METHODS or "plv" in METHODS:
    print("⚠️  PLV/wPLI lenti (O(N²)/trial) — stima ~1-2 ore su dataset completo.")
    print("   Per analisi rapida usa solo 'pcc'.")

# ---- Accumulatori ----
results = {m: {"sum": np.zeros((N_CHANS, N_CHANS)), "vals": [], "n": 0}
           for m in METHODS}

for i, csv_path in enumerate(tqdm(_trial_paths, desc="Analisi connettività")):
    x_np = load_csv_trial(csv_path)
    for m in METHODS:
        mat = CONN_FN_ANALYSIS[m](x_np)
        results[m]["sum"] += mat
        results[m]["n"]   += 1
        if i % VAL_SUBSAMPLE == 0:
            results[m]["vals"].extend(
                mat[np.triu_indices(N_CHANS, k=1)].tolist()
            )

for m in METHODS:
    results[m]["avg"]  = results[m]["sum"] / max(results[m]["n"], 1)
    results[m]["vals"] = np.array(results[m]["vals"])
    results[m]["conn"] = results[m]["avg"].sum(axis=1)

print(f"\n✅ Analisi completata — {results[METHODS[0]]['n']} trial, "
      f"vals campionati ogni {VAL_SUBSAMPLE} trial")

# ---- Nomi canali ----
_eloc_candidates = [
    project_root / "data" / "interim" / "ebneuro.csv",
    Path("/mnt/c/Users/students/Desktop/Paolo/LM_Thesis/ebneuro.csv"),
]
_ch_names = [str(i) for i in range(N_CHANS)]
for _p in _eloc_candidates:
    if _p.exists():
        _ch_names = pd.read_csv(_p, sep=";", decimal=",")["labels"].tolist()[:N_CHANS]
        print(f"Nomi canali caricati da {_p.name}")
        break

# ---- Plot: 3 righe × n_methods colonne ----
n_methods = len(METHODS)
fig, axes = plt.subplots(3, n_methods, figsize=(6 * n_methods, 14))
if n_methods == 1:
    axes = axes[:, np.newaxis]
fig.suptitle(
    f"Analisi Connettività — {n_trials_analysis} trial "
    f"({len(_subjects_all)} sogg × {len(_sessions_all)} sess × {len(_words_all)} parole)",
    fontsize=13
)

for col, m in enumerate(METHODS):
    avg   = results[m]["avg"]
    conn  = results[m]["conn"]
    vals  = results[m]["vals"]
    mean_c, std_c = conn.mean(), conn.std()

    sns.heatmap(avg, ax=axes[0, col], cmap="RdYlBu_r",
                vmin=0, vmax=float(np.percentile(vals, 95)) if len(vals) else 1.0,
                xticklabels=False, yticklabels=False)
    axes[0, col].set_title(f"{m.upper()} — matrice media\n({n_trials_analysis} trial)")

    bar_colors = [
        "red"     if c < mean_c - 2 * std_c else
        "orange"  if c < mean_c -     std_c else
        "steelblue"
        for c in conn
    ]
    axes[1, col].bar(range(N_CHANS), conn, color=bar_colors)
    axes[1, col].axhline(mean_c,             color="black",  ls="--", lw=1.2,
                         label=f"μ={mean_c:.3f}")
    axes[1, col].axhline(mean_c -     std_c, color="orange", ls="--", lw=1.2,
                         label=f"μ-1σ={mean_c - std_c:.3f}")
    axes[1, col].axhline(mean_c - 2 * std_c, color="red",    ls="--", lw=1.2,
                         label=f"μ-2σ={mean_c - 2*std_c:.3f}")
    axes[1, col].set_xlabel("Canale (idx)")
    axes[1, col].set_ylabel("Connettività totale")
    axes[1, col].set_title(f"{m.upper()} — per canale\n🔴<μ-2σ  🟠<μ-1σ")
    axes[1, col].legend(fontsize=7)

    if len(vals):
        axes[2, col].hist(vals, bins=100, color="steelblue", alpha=0.7, edgecolor="none")
        for pct in [25, 50, 75]:
            thr = float(np.percentile(vals, pct))
            axes[2, col].axvline(thr, color="red", ls="--", lw=1.2,
                                 label=f"p{pct}={thr:.3f}")
        axes[2, col].set_xlabel(f"|{m.upper()}|")
        axes[2, col].set_ylabel("Frequenza")
        axes[2, col].set_title(
            f"{m.upper()} — distribuzione archi\n"
            f"(guida EDGE_THRESHOLD, campionato 1/{VAL_SUBSAMPLE} trial)")
        axes[2, col].legend(fontsize=7)

plt.tight_layout()
fig_path = project_root / "figures" / "eeg07e_connectivity_analysis_full.png"
fig.savefig(fig_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Plot salvato: {fig_path}")

# ---- Riepilogo testuale ----
for m in METHODS:
    conn   = results[m]["conn"]
    vals   = results[m]["vals"]
    mean_c = conn.mean(); std_c = conn.std()
    print(f"\n{'='*50}  {m.upper()}")
    print(f"  N trial analizzati : {results[m]['n']}")
    print(f"  Connettività media : μ={mean_c:.4f}  σ={std_c:.4f}")
    weak = np.where(conn < mean_c - 2 * std_c)[0]
    if len(weak):
        names_str = ", ".join(f"idx={i} ({_ch_names[i]})" for i in weak)
        print(f"  ⚠️  Canali deboli (<μ-2σ={mean_c-2*std_c:.4f}): {names_str}")
        print(f"     → candidati per PRUNE_CHANNEL_NAMES in cell 12")
    else:
        print(f"  ✅ Nessun canale debole (<μ-2σ)")
    if len(vals):
        print(f"  PRUNE_EDGE_THRESHOLD consigliato:")
        for pct in [25, 50, 75]:
            thr  = float(np.percentile(vals, pct))
            kept = float((vals >= thr).mean() * 100)
            print(f"    p{pct:2d} = {thr:.4f}  →  mantiene ~{kept:.0f}% archi")

In [ ]:
# ============================================================
# 10. PARAMETRI PRUNING — AUTO-ESTRATTI DALL'ANALISI (cell 9)
# ============================================================

if "results" not in globals() or "conn" not in results.get(METHODS[0], {}):
    raise RuntimeError(
        "❌ 'results' non definito — esegui prima cell 9 (Analisi Connettività)."
    )

REF_METHOD = METHODS[0]
conn_ref   = results[REF_METHOD]["conn"]
vals_ref   = results[REF_METHOD]["vals"]
mean_c     = conn_ref.mean()
std_c      = conn_ref.std()

# 1. CANALI DA RIMUOVERE: sotto μ - 2σ
weak_idx           = np.where(conn_ref < mean_c - 2 * std_c)[0]
PRUNE_CHANNEL_NAMES = [_ch_names[i] for i in weak_idx]

# 2. SOGLIA ARCHI: percentile scelto (0 = nessun pruning archi)
USE_PERCENTILE = 50   # 0 | 25 | 50 | 75

if USE_PERCENTILE > 0 and len(vals_ref) > 0:
    PRUNE_EDGE_THRESHOLD = float(np.percentile(vals_ref, USE_PERCENTILE))
else:
    PRUNE_EDGE_THRESHOLD = 0.0

print("=" * 55)
print(f"PARAMETRI AUTO-ESTRATTI (Riferimento: {REF_METHOD.upper()})")
print("=" * 55)
if len(PRUNE_CHANNEL_NAMES) > 0:
    print(f"🔴 Canali deboli (< μ-2σ): {len(PRUNE_CHANNEL_NAMES)}")
    print(f"   PRUNE_CHANNEL_NAMES = {PRUNE_CHANNEL_NAMES}")
else:
    print("✅ Nessun canale debole (<μ-2σ).")

if USE_PERCENTILE > 0:
    print(f"✂️  Soglia archi (p{USE_PERCENTILE}): {PRUNE_EDGE_THRESHOLD:.4f}")
else:
    print("✅ Nessun pruning archi (USE_PERCENTILE=0).")
print("=" * 55)

In [ ]:
# ============================================================
# BUILD TENSORI PRUNATI
# Carica i .pt esistenti, applica PRUNE_EDGE_THRESHOLD +
# PRUNE_CHANNEL_NAMES e salva nuovi file.
#
# Rispetta FORCE_REBUILD: se False e il file prunato esiste già, salta.
# Output: data/interim/graphs/{name}_drop*_thr*.pt
# ============================================================

# Converti nomi canali → indici da tenere
_prune_drop_idx = set()
for _name in PRUNE_CHANNEL_NAMES:
    if _name in _ch_names:
        _prune_drop_idx.add(_ch_names.index(_name))
    else:
        print(f"⚠️  Canale '{_name}' non trovato in _ch_names — saltato")

_pruned_keep = [i for i in range(N_CHANS) if i not in _prune_drop_idx]
N_PRUNED     = len(_pruned_keep)

print(f"Canali rimossi : {[_ch_names[i] for i in _prune_drop_idx] or 'nessuno'}")
print(f"Canali rimasti : {N_PRUNED}/{N_CHANS}")
print(f"Edge threshold : {PRUNE_EDGE_THRESHOLD}")

if N_PRUNED == N_CHANS and PRUNE_EDGE_THRESHOLD == 0.0:
    print("\n⚠️  Nessun pruning attivo (PRUNE_EDGE_THRESHOLD=0 e PRUNE_CHANNEL_NAMES=[]).")
    print("   Imposta i parametri in cell 10 e riesegui.")
else:
    # Suffisso per il nome file
    _suf_ch  = ("_drop" + "_".join(PRUNE_CHANNEL_NAMES)) if PRUNE_CHANNEL_NAMES else ""
    _suf_thr = f"_thr{PRUNE_EDGE_THRESHOLD:.3f}" if PRUNE_EDGE_THRESHOLD > 0 else ""
    _suffix  = _suf_ch + _suf_thr

    for pt_file in sorted(GRAPHS_DIR.glob("*.pt")):
        # Salta file già prunati
        if "_drop" in pt_file.name or "_thr" in pt_file.name:
            continue

        # Calcola out_path PRIMA di caricare — per poter skippare
        is_hgraph = pt_file.name.startswith("hgraph")
        _k        = int(pt_file.stem.split("_k")[-1]) if "_k" in pt_file.stem else K_HYPER
        out_path  = GRAPHS_DIR / (pt_file.stem + _suffix + ".pt")

        if out_path.exists() and not FORCE_REBUILD:
            print(f"Skip (esiste): {out_path.name}")
            continue

        dl = torch.load(pt_file, weights_only=False)

        print(f"\nPruning {pt_file.name} → {out_path.name} ...")
        pruned_list = []

        for d in tqdm(dl, desc=out_path.name, leave=False):
            x_np     = d.x.numpy()                          # (N_CHANS, 384)
            x_pruned = x_np[_pruned_keep, :]                # (N_PRUNED, 384)

            if not is_hgraph:
                if PRUNE_EDGE_THRESHOLD > 0.0:
                    # Ricalcola grafo da segnale prunato con threshold
                    _mat = pcc_matrix(x_pruned)
                    _ei  = knn_edge_index(_mat, k=_k,
                                          threshold=PRUNE_EDGE_THRESHOLD)
                else:
                    # Riproietta edge_index: rimuovi archi verso canali eliminati
                    _idx_map = {old: new for new, old in enumerate(_pruned_keep)}
                    _old_ei  = d.edge_index.numpy()
                    _mask    = [
                        (int(s) in _idx_map and int(t) in _idx_map)
                        for s, t in zip(_old_ei[0], _old_ei[1])
                    ]
                    _new_src = [_idx_map[int(s)]
                                for s, ok in zip(_old_ei[0], _mask) if ok]
                    _new_dst = [_idx_map[int(t)]
                                for t, ok in zip(_old_ei[1], _mask) if ok]
                    _ei      = torch.tensor([_new_src, _new_dst],
                                            dtype=torch.long)

                pruned_list.append(Data(
                    x          = torch.tensor(x_pruned, dtype=torch.float32),
                    edge_index = _ei,
                    y          = d.y,
                    label_id   = d.label_id,
                    subj       = d.subj,
                    sess       = d.sess,
                ))
            else:
                # Ipergrafo: ricalcola hyperedge_index sui canali prunati
                _he = hyperedge_index_fn(x_pruned, k=_k,
                                         threshold=PRUNE_EDGE_THRESHOLD)
                pruned_list.append(Data(
                    x               = torch.tensor(x_pruned, dtype=torch.float32),
                    hyperedge_index = _he,
                    num_hyperedges  = int(_he[1].max()) + 1,
                    y               = d.y,
                    label_id        = d.label_id,
                    subj            = d.subj,
                    sess            = d.sess,
                ))

        torch.save(pruned_list, out_path)
        print(f"  ✅ {out_path.name}: {len(pruned_list)} oggetti, "
              f"x.shape={pruned_list[0].x.shape}")

In [ ]:
# ============================================================
# VISUALIZZAZIONE INTERATTIVA: Grafo vs Ipergrafo
# Scegli soggetto, parola, sessione, k dal menu
# ============================================================

import ipywidgets as widgets
from ipywidgets import interact, interactive_output, VBox, HBox
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import numpy as np
import networkx as nx
import pandas as pd
from scipy.spatial import ConvexHull

# ---- Carica posizioni elettrodi da ebneuro.csv ----
_eloc_candidates = [
    project_root / "data" / "interim" / "ebneuro.csv",
    Path("/mnt/c/Users/students/Desktop/Paolo/LM_Thesis/ebneuro.csv"),
]
_eloc_df = None
for _p in _eloc_candidates:
    if _p.exists():
        _eloc_df = pd.read_csv(_p, sep=";", decimal=",")
        break

def get_positions(n_nodes):
    """Posizioni 2D elettrodi: topomap se disponibile, altrimenti circolare."""
    if _eloc_df is not None and len(_eloc_df) >= n_nodes:
        df = _eloc_df.iloc[:n_nodes]
        theta_deg = df["theta"].values.astype(float)
        radius    = df["radius"].values.astype(float)
        theta_rad = np.deg2rad(theta_deg)
        x = radius * np.sin(theta_rad)
        y = radius * np.cos(theta_rad)
        return np.stack([x, y], axis=1)
    angles = np.linspace(0, 2*np.pi, n_nodes, endpoint=False)
    return np.stack([np.cos(angles), np.sin(angles)], axis=1)

# ---- Widgets ----
_subjects  = sorted(set(d.name.split("_")[0] for d in CSV_ROOT.iterdir() if d.is_dir()))
_words     = sorted(word2labelid.keys())
_sessions  = sorted(set(d.name.split("_")[1] for d in CSV_ROOT.iterdir() if d.is_dir()))

w_subj   = widgets.Dropdown(options=_subjects, value=_subjects[0], description="Soggetto:")
w_word   = widgets.Dropdown(options=_words,    value=_words[0],    description="Parola:")
w_sess   = widgets.Dropdown(options=_sessions, value=_sessions[0], description="Sessione:")
w_k      = widgets.IntSlider(min=2, max=15, value=6, step=1,       description="k vicini:")
w_he_max = widgets.IntSlider(min=5, max=61, value=20, step=1,      description="Max iperedge:")

def draw_convex_blob(ax, pos, members, color, alpha=0.25, pad=0.04):
    """Disegna blob convex hull attorno ai nodi dell'iperedge."""
    pts = pos[members]
    if len(pts) < 3:
        cx, cy = pts.mean(axis=0)
        circle = plt.Circle((cx, cy), pad*3, color=color, alpha=alpha, zorder=1)
        ax.add_patch(circle)
        return
    try:
        hull = ConvexHull(pts)
        hull_pts = pts[hull.vertices]
        centroid = hull_pts.mean(axis=0)
        expanded = centroid + (hull_pts - centroid) * (
            1 + pad / (np.linalg.norm(hull_pts - centroid, axis=1).mean() + 1e-9))
        poly = plt.Polygon(np.vstack([expanded, expanded[0]]),
                           closed=True, color=color, alpha=alpha, zorder=1,
                           linewidth=1.2, edgecolor=color)
        ax.add_patch(poly)
    except Exception:
        pass

def plot_comparison(subj, word, session, k, max_he):
    csv_path = CSV_ROOT / f"{subj}_{session}" / f"{word}_img.csv"
    if not csv_path.exists():
        print(f"File non trovato: {csv_path}")
        return

    x_np = load_csv_trial(csv_path)
    N    = x_np.shape[0]
    pos  = get_positions(N)

    pcc = np.abs(np.corrcoef(x_np)); np.fill_diagonal(pcc, 0.0)
    ei  = knn_edge_index(pcc, k=k).numpy()

    he  = hyperedge_index_fn(x_np, k=k).numpy()
    n_he = int(he[1].max()) + 1
    he_members = {}
    for v, e in zip(he[0], he[1]):
        he_members.setdefault(int(e), []).append(int(v))

    G = nx.Graph()
    G.add_nodes_from(range(N))
    for s, t in zip(ei[0], ei[1]):
        if s < t:
            G.add_edge(int(s), int(t), weight=float(pcc[s, t]))
    node_deg = np.array([G.degree(i) for i in range(N)], dtype=float)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle(f"Soggetto {subj}  |  Parola '{word}'  |  Sessione {session}  |  k={k}",
                 fontsize=13, fontweight="bold")

    ax1.set_title(f"Grafo k-NN  ({G.number_of_edges()} archi)", fontsize=11)
    edge_weights = [pcc[u, v] for u, v in G.edges()]
    nx.draw_networkx_edges(G, pos={i: pos[i] for i in range(N)},
                           ax=ax1, edge_color=edge_weights,
                           edge_cmap=cm.Blues, width=1.5, alpha=0.7)
    sc1 = ax1.scatter(pos[:, 0], pos[:, 1],
                      s=60 + node_deg * 20,
                      c=node_deg, cmap="plasma", zorder=5,
                      edgecolors="black", linewidths=0.5)
    plt.colorbar(sc1, ax=ax1, label="Grado nodo")
    ax1.add_patch(plt.Circle((0, 0), 0.6, fill=False, color="gray",
                              linewidth=1.5, linestyle="--"))
    ax1.set_aspect("equal"); ax1.axis("off")

    ax2.set_title(f"Ipergrafo  ({n_he} iperedge, mostra prime {min(max_he, n_he)})", fontsize=11)
    colors = cm.tab20(np.linspace(0, 1, min(max_he, n_he)))
    for (e_id, members), color in zip(list(he_members.items())[:max_he], colors):
        draw_convex_blob(ax2, pos, members, color=color[:3], alpha=0.3)
    n_he_per_node = np.bincount(he[0], minlength=N)
    sc2 = ax2.scatter(pos[:, 0], pos[:, 1],
                      s=60 + n_he_per_node * 5,
                      c=n_he_per_node, cmap="YlOrRd", zorder=5,
                      edgecolors="black", linewidths=0.5)
    plt.colorbar(sc2, ax=ax2, label="N° iperedge per nodo")
    ax2.add_patch(plt.Circle((0, 0), 0.6, fill=False, color="gray",
                              linewidth=1.5, linestyle="--"))
    ax2.set_aspect("equal"); ax2.axis("off")

    plt.tight_layout()
    plt.show()

# ---- Render: interactive_output evita doppio plot ----
out = interactive_output(
    plot_comparison,
    {"subj": w_subj, "word": w_word, "session": w_sess,
     "k": w_k, "max_he": w_he_max}
)
display(VBox([
    HBox([w_subj, w_word, w_sess]),
    HBox([w_k, w_he_max]),
    out
]))